# Rally 12B Heretic

Runs upstream `heretic-llm` against `google/gemma-4-12B-it`, saves a merged checkpoint, and uploads it to Hugging Face when `HF_TOKEN` is available.

Set Kaggle accelerator to **GPU A100** and Internet to **On** before running.

In [ ]:
import os, platform, shutil
from pathlib import Path

LABEL = os.environ.get('HERETIC_PROOF_LABEL', 'rally-12b')
REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
WORKING = Path('/kaggle/working')
REPO_DIR = WORKING / 'heretic-to-onnx'
print({'label': LABEL, 'repo_url': REPO_URL, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

In [ ]:
import torch, shutil, platform, os
from pathlib import Path

print('python_platform=', platform.platform())
print('cuda_available=', torch.cuda.is_available())
print('gpu_count=', torch.cuda.device_count())
gpu_names = []
min_vram_gb = float(os.environ.get('RALLY_12B_MIN_VRAM_GB', '14'))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpu_names.append(p.name)
    vram_gb = round(p.total_memory / 1024**3, 2)
    print(f'gpu_{i}=', p.name, vram_gb, 'GiB')
    if vram_gb < min_vram_gb:
        raise RuntimeError(f'GPU {i} has {vram_gb} GiB; need >= {min_vram_gb} GiB for 12B Heretic.')
usage = shutil.disk_usage(Path('/kaggle/working'))
print('working_disk_free_gb=', round(usage.free / 1024**3, 2))
if not any('T4' in name or 'A100' in name or 'H100' in name for name in gpu_names):
    raise RuntimeError(f'Expected T4/A100/H100, got {gpu_names}. Push with --accelerator GPU_T4_x2.')
os.environ['RALLY_12B_A100_MODE'] = '1' if any(round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2) >= 38 for i in range(torch.cuda.device_count())) else '0'
print('rally_12b_a100_mode=', os.environ['RALLY_12B_A100_MODE'])

In [ ]:
import subprocess, sys
transformers_ref = 'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af'
packages = [
    'heretic-llm',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-deps', transformers_ref])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
import transformers
print('transformers=', transformers.__version__)

In [ ]:
import os, subprocess
from pathlib import Path

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ.setdefault('HF_TOKEN', secrets.get_secret('HF_TOKEN'))
    try:
        os.environ.setdefault('HF_MERGED_REPO_ID', secrets.get_secret('HF_MERGED_REPO_ID'))
    except Exception:
        pass
except Exception:
    pass

if 'HF_TOKEN' in os.environ:
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('HF login ok')
else:
    print('HF_TOKEN not set; running without Hub upload')

In [ ]:
import os
UPLOAD_ARGS = ''
os.environ.setdefault('HF_MERGED_REPO_ID', os.environ.get('RALLY_12B_HERETIC_REPO', 'thomasjvu/rally-12b-heretic-merged'))
if os.environ.get('HF_TOKEN') and os.environ.get('HF_MERGED_REPO_ID'):
    UPLOAD_ARGS = f"--upload-merged-to {os.environ['HF_MERGED_REPO_ID']}"
a100_mode = os.environ.get('RALLY_12B_A100_MODE', '0') == '1'
t4x2_mode = torch.cuda.device_count() >= 2 and not a100_mode
accelerator = 'a100' if a100_mode else ('t4x2' if t4x2_mode else 'single-gpu')
n_trials = 20 if a100_mode else 12
prompt_rows = 160 if a100_mode else 80
eval_rows = 80 if a100_mode else 40
print('heretic_profile=', {'accelerator': accelerator, 'n_trials': n_trials, 'prompt_rows': prompt_rows, 'eval_rows': eval_rows})

!python {REPO_DIR}/scripts/kaggle_heretic_2b_proof.py \
  --label {LABEL} \
  --accelerator {accelerator} \
  --n-trials {n_trials} \
  --n-startup-trials 8 \
  --prompt-rows {prompt_rows} \
  --eval-rows {eval_rows} \
  --max-response-length 64 \
  {UPLOAD_ARGS}